In [28]:
from itertools import dropwhile
import random
import traceback

import requests


class Vocab:
    def __init__(self, reading, furigana, meaning, level, pos, sentenceEN, sentenceJP, vocabs):
        self.reading = reading
        self.furigana = furigana
        self.meaning = meaning
        self.level = level
        self.pos = pos
        self.sentenceEN = sentenceEN
        self.sentenceJP = sentenceJP
        self.vocabs = vocabs
        
    def __eq__(self, other):
        return self.reading == other.reading
    
    def __str__(self):
        strg = ""
        strg += f"{self.reading} {self.furigana}\n"
        strg += f"{self.meaning}\n"
        strg += f"JLPT-N{self.level} {self.pos}\n"
        strg += f"{self.sentenceEN}\n"
        strg += f"{self.sentenceJP}\n"
        
        
        strg += "-----\n"
        strg += ", ".join([v.reading for v in self.vocabs])
            
        return strg
    def __hash__(self):
        return hash(self.reading)
    
    def GetData(self):
        return [self.reading, self.furigana, self.meaning, self.level, self.pos, self.sentenceEN, self.sentenceJP]
        


def GetWord(query: str, max_results: int = 5):
    url = f"https://jisho.org/api/v1/search/words?keyword={query}"

    try:
        res = requests.get(url)
        res.raise_for_status()
        data = res.json()
    except requests.RequestException as e:
        print(f"Error calling Jisho API: {e}")
        return

    results = data.get("data", [])
    
    newr = []
    passed = False
    for result in results:
        if any(char.isdigit() for char in query) or (not any(char.isdigit() for char in result["slug"])):
            newr.append(result)
    results = newr
        
    if not results:
        print(f"No results found for '{query}'.")
        return None
    return results

def GetSentences(query: str, reverse : bool = False):
    base = f"https://tatoeba.org/en/api_v0/search?query={query}&from=jpn&to=eng"
    
    if reverse:
        base = f"https://tatoeba.org/en/api_v0/search?query={query}&from=eng&to=jpn"

    try:
        res = requests.get(base)
        res.raise_for_status()
        data = res.json()
    except requests.RequestException as e:
        print(f"Error calling Tatoeba API: {e}")
        return
    
    sentences = data.get("results", {})
    
    if len(sentences) == 0:
        if reverse is False:
            return GetSentences(query, True)
        return "",""

    try:
        if reverse is True:
            return sentences[0]["translations"][0][0]["text"], sentences[0]["text"]
        return sentences[0]["text"], sentences[0]["translations"][0][0]["text"]
    except:
        if reverse is False:
            return GetSentences(query, True)
        return "",""


In [29]:
import pykakasi
import re
import traceback

def get_html_furigana(text):
    kks = pykakasi.kakasi()
    result = kks.convert(text)
    
    html_output = ""
    
    for item in result:
        kanji_block = item['orig']
        reading = item['hira']
        
        if kanji_block == reading:
            html_output += kanji_block
            continue

        # Logic to handle Okurigana (like the 'き' in '生きる')
        # We find where the trailing hiragana starts
        match = re.search(r'([ぁ-ん]+)$', kanji_block)
        
        if match:
            okurigana = match.group(1)
            # Remove okurigana from the end of both the kanji block and the reading
            base_kanji = kanji_block[:-len(okurigana)]
            base_reading = reading[:-len(okurigana)]
            
            html_output += f"<ruby>{base_kanji}<rt>{base_reading}</rt></ruby>{okurigana}"
        else:
            # No okurigana found (pure kanji word like 日本語)
            html_output += f"<ruby>{kanji_block}<rt>{reading}</rt></ruby>"
            
    return html_output

def CreateVocab(data, onHiragana, kanji, hiragana):
    
    
    reading = kanji  if not onHiragana else hiragana
    
    if any(v.reading == reading for v in AddedVocab):
        #print(f"{reading} already exists")
        vocab = next((v for v in AddedVocab if v.reading == reading), None)
        if len(vocab.vocabs) == 0:
            references = CreateReferences(data)
            vocab.vocabs = references
            #print(f"Added +{len(references)} relations")
    else:
        furigana = hiragana
    
        try:
            jlpt = data[0]["jlpt"]
            jlpt = str(sorted(map(lambda a: int(a[-1]), jlpt), reverse=True)[0])
        except:
            jlpt = "?"
            
        meaning = ", ".join(data[0]["senses"][0]["english_definitions"])
        pos = data[0]["senses"][0]["parts_of_speech"][0]
        
        sentJP, sentEN = GetSentences(kanji)
        
        references = CreateReferences(data)
        
        vocab = Vocab(reading, furigana, meaning, jlpt, pos, sentEN, sentJP, references)
    
    return vocab
     
def CreateReferences(data):
    references = []
    for result in data[1:]:
        try:
            readingH, hiraganaH = result["slug"], result["japanese"][0]["reading"]
            
            reading = data[0]["slug"]
            
            if reading not in readingH:
                continue
                
            if any(char.isdigit() for char in readingH):
                continue
            
            
            meaningH = ", ".join(result["senses"][0]["english_definitions"])
            posH = result["senses"][0]["parts_of_speech"][0]
            sentJPH, sentENH = GetSentences(readingH)
            if sentJPH == "":
                continue

    
            try:
                jlptH = result["jlpt"]
                jlptH = str(sorted(map(lambda a: int(a[-1]), jlptH), reverse=True)[0])
                
            except:
                jlptH = "?"
                
            if any(v.reading == readingH for v in AddedVocab):
                print(f"{readingH} already exists")
                reference = next((v for v in AddedVocab if v.reading == readingH), None)
            else:
                reference = Vocab(readingH, hiraganaH, meaningH, jlptH, posH, sentENH, sentJPH, [])
                
            references.append(reference)
        except Exception:
            # print("Error: " + readingH)
            # print(traceback.format_exc())
            continue
        if len(references) >= 3:
            break
    return references
    

In [30]:
import csv
import os

def append_to_csv(file_name, data):

    # Check if the file exists to determine if we need a header (optional)
    file_exists = os.path.isfile(file_name)
    
    # Opening in 'a' (append) mode. 'newline=""' prevents blank rows on Windows.
    with open(file_name, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file, delimiter=';')
        writer.writerow(data)


In [31]:
import json
from tqdm.notebook import tqdm
AddedVocab = []
# 1. Load the data
with open('JLPT N5.json', 'r', encoding='utf-8') as f:
    j = json.load(f)

# 2. Loop through the elements
for i, entry in tqdm(enumerate(j), desc="Progress: "):
    kanji = entry["Kanji"]
    onHiragana = False
    if kanji == "":
        kanji = entry["Hiragana"]
        onHiragana = True
        
    vocab = CreateVocab(GetWord(kanji), onHiragana, entry["Kanji"], entry["Hiragana"])
    
    AddedVocab.append(vocab)
    
    for v in vocab.vocabs:
        
        AddedVocab.append(v)
       
  
        


Progress: : 0it [00:00, ?it/s]

上げる already exists


KeyboardInterrupt: 

In [32]:
used = set()

for v in AddedVocab:
    if v not in used:
        used.add(v)
        append_to_csv("vocab.csv", v.GetData())
 
used = set()   
for v in AddedVocab:
    for r in v.vocabs:
        if v not in used:
            used.add((r,v))
            append_to_csv("relation.csv", [v.reading, r.reading])